In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "kano2011visual")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "2011VisRes.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="kano2011visual"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [3]:
df.rename(columns={"subject": "ape",
    "group":"group_original",
    "*gap-overlap study":"gap_overlap_study"}, inplace=True)

In [4]:
df = df[~df['spe'].isin(['human'])]

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='ape', right_on='name', how='left')

In [6]:
# df.columns
df['spe'].replace('chimp', 'chimpanzee', inplace=True, regex=True)

spe_2=[]
for index, row in df.iterrows():
    if not pd.isna(row['species']):
        spe_2.append(row["species"])
    else:
        spe_2.append(row['spe'])
df = df.assign(species=spe_2)

df.rename(columns={"ape": "participant", 
                   'group_original':"group_id"}, inplace=True)


In [7]:
kano2011visual_standardized=df[['study_id',  'participant',   'sex', 'species','group_id','gap', 'overlap']]
comp_out_path_stand = os.path.join(out_pathway, 'kano2011visual_standardized.csv')
kano2011visual_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =kano2011visual_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
kano2011visual_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'kano2011visual_glossary.csv')
kano2011visual_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
